In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd


In [35]:
def create_webdriver():
    opts = Options()
    # opts.add_argument("--headless=new")  # enable for no-GUI scraping
    # opts.add_experimental_option("detach", True)  # keep window open after script ends
    return webdriver.Chrome(options=opts)


WEBSITE = "https://old.reddit.com/r/wallstreetbets/"
# https://old.reddit.com/r/wallstreetbets/
# https://books.toscrape.com/
driver = create_webdriver()
driver.get(WEBSITE)


In [36]:
# Wait until the project links are present
titles = []
post_idList = []
created_utcList = []
flairsList = []
upvotesList = []
num_commentsList = []
permalinkList = []

i = 0

while i < 3:
    wait = WebDriverWait(driver, 10)
    things = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.thing"))
    )




    for thing in things:
        if "stickied" in thing.get_attribute("class"):
            continue
        if thing.get_attribute("data-promoted") == "true":
            continue
        title = thing.find_element(By.CSS_SELECTOR, 'p.title > a')
        titles.append(title.text)

        post_id = thing.get_attribute("data-fullname")
        post_idList.append(post_id)

        time_elements = thing.find_elements(By.TAG_NAME, "time")
        if time_elements:
            created_utc = time_elements[0].get_attribute("datetime")
        else:
            created_utc = None
        created_utcList.append(created_utc)

        flairs = thing.find_element(By.CSS_SELECTOR, 'span.linkflairlabel')
        flairsList.append(flairs.text)

        upvotes = thing.find_element(By.CSS_SELECTOR, 'div.score')
        upvotesList.append(upvotes.text)

        comments = thing.find_element(By.CSS_SELECTOR, 'a.comments')
        num_commentsList.append(comments.text)

        permalink = comments.get_attribute("href")
        permalinkList.append(permalink)

        


    
    next_button = WebDriverWait(driver, 10).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'span.next-button')))
    if next_button.get_attribute('Disabled'):
        break  # Exit the loop if the next button is disabled
    else:
        # Click the next button to navigate to the next page
        next_button.click()
    i += 1


driver.quit()


In [39]:
df = pd.DataFrame({'titles': titles, 'post_idList': post_idList, 'created_utc': created_utcList, 'flairs': flairsList, 'upvotes': upvotesList, 'num_comments': num_commentsList, 'permalink': permalinkList})
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 500)  # adjust for your screen
pd.set_option('display.max_columns', None)
print(df)

                                                                                                                                          titles post_idList                created_utc            flairs upvotes    num_comments                                                                                                     permalink
0                                                                                                                                   WE DID IT!!!  t3_1mlyk81  2025-08-09T19:53:00+00:00              Gain            334 comments                                           https://old.reddit.com/r/wallstreetbets/comments/1mlyk81/we_did_it/
1                    BofA flagged 26 companies at risk because of AI. This is the Bloomberg chart showing their performance relative to the S&P.  t3_1mlxc4f  2025-08-09T19:01:33+00:00        Discussion            222 comments     https://old.reddit.com/r/wallstreetbets/comments/1mlxc4f/bofa_flagged_26_companies_at_risk_because

# Next Steps:
- data cleaning
- perform analytics
- visualizations